In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates


In [4]:

def calculate_metrics(trade_log, initial_demat, final_demat, equity_curve):
    """Calculates performance metrics safely, handling empty logs and NaNs."""
    
    # 1. Initialize with SAFEST defaults (prevent KeyError)
    metrics = {
        'initial_demat': initial_demat,
        'final_demat': final_demat if not np.isnan(final_demat) else initial_demat,
        'num_trades': 0,
        'wins': 0,
        'losses': 0,
        'win_rate': 0.0,
        'avg_win': 0.0,
        'avg_loss': 0.0,
        'median_win': 0.0,
        'median_loss': 0.0,
        'avg_return_pct': 0.0,
        'total_winning': 0.0,
        'total_losing': 0.0,
        'profit_factor': 0.0,
        'expectancy': 0.0,
        'largest_single_trade_gain': 0.0,
        'largest_single_trade_loss': 0.0,
        'avg_holding_days': 0.0,
        'median_holding_days': 0.0,
        'SL_exit_rate': 0.0,
        'Time_exit_rate': 0.0,
        'total_return': 0.0,
        'annualized_return': 0.0,
        'annualized_volatility': 0.0,
        'sharpe_ratio': 0.0,
        'max_drawdown': 0.0
    }

    # 2. Handle Edge Case: No Trades
    if trade_log.empty:
        if not equity_curve.empty:
            # Calculate drawdown even if flat
            peak = equity_curve.cummax()
            # Avoid division by zero if peak is 0
            drawdown = (equity_curve - peak) / peak.replace(0, 1) 
            metrics['max_drawdown'] = drawdown.min()
            metrics['total_return'] = (metrics['final_demat'] / initial_demat) - 1
        return metrics

    # 3. Calculate Stats (Safe Division)
    wins = trade_log[trade_log['pnl'] > 0]
    losses = trade_log[trade_log['pnl'] <= 0]

    metrics['num_trades'] = len(trade_log)
    metrics['wins'] = len(wins)
    metrics['losses'] = len(losses)
    metrics['win_rate'] = (metrics['wins'] / metrics['num_trades']) if metrics['num_trades'] > 0 else 0
    
    metrics['avg_win'] = wins['pnl'].mean() if not wins.empty else 0
    metrics['avg_loss'] = losses['pnl'].mean() if not losses.empty else 0
    metrics['median_win'] = wins['pnl'].median() if not wins.empty else 0
    metrics['median_loss'] = losses['pnl'].median() if not losses.empty else 0
    
    metrics['total_winning'] = wins['pnl'].sum()
    metrics['total_losing'] = abs(losses['pnl'].sum())
    
    # Safe Profit Factor
    if metrics['total_losing'] > 0:
        metrics['profit_factor'] = metrics['total_winning'] / metrics['total_losing']
    else:
        metrics['profit_factor'] = np.inf if metrics['total_winning'] > 0 else 0

    metrics['expectancy'] = (metrics['win_rate'] * metrics['avg_win']) + ((1 - metrics['win_rate']) * metrics['avg_loss'])
    
    metrics['largest_single_trade_gain'] = wins['pnl'].max() if not wins.empty else 0
    metrics['largest_single_trade_loss'] = losses['pnl'].min() if not losses.empty else 0
    
    metrics['avg_return_pct'] = trade_log['return_pct'].mean()
    metrics['avg_holding_days'] = trade_log['holding_days'].mean()
    metrics['median_holding_days'] = trade_log['holding_days'].median()
    
    metrics['SL_exit_rate'] = len(trade_log[trade_log['exit_reason'] == 'SL']) / metrics['num_trades']
    metrics['Time_exit_rate'] = len(trade_log[trade_log['exit_reason'] == 'TIME']) / metrics['num_trades']

    metrics['total_return'] = (metrics['final_demat'] / initial_demat) - 1
    
    # 4. Equity Curve Metrics
    if not equity_curve.empty:
        days_span = (equity_curve.index.max() - equity_curve.index.min()).days
        days_span = max(days_span, 1) # Avoid division by zero
        
        # Simple annualization safety
        if days_span < 365:
            metrics['annualized_return'] = metrics['total_return'] # Fallback for short periods
        else:
            metrics['annualized_return'] = (1 + metrics['total_return']) ** (365.0 / days_span) - 1

        daily_returns = equity_curve.pct_change().dropna()
        metrics['annualized_volatility'] = daily_returns.std() * np.sqrt(252)
        
        if metrics['annualized_volatility'] > 0:
            metrics['sharpe_ratio'] = metrics['annualized_return'] / metrics['annualized_volatility']
        
        peak = equity_curve.cummax()
        drawdown = (equity_curve - peak) / peak.replace(0, 1)
        metrics['max_drawdown'] = drawdown.min()
    
    return metrics


def plot_backtest(result, df, figures_dir='figures'):
    """Safely plots backtest results."""
    
    if not os.path.exists(figures_dir):
        os.makedirs(figures_dir)
        
    log = result['trade_log']
    equity_mtm = result['equity_curve']
    equity_no_mtm = result['equity_curve_no_mtm']
    price_col = result['price_col_used']
    
    # --- FIX: Ensure DataFrame has DatetimeIndex for Plotting ---
    if not isinstance(df.index, pd.DatetimeIndex):
        if 'Date' in df.columns:
            df = df.set_index('Date')
            df.index = pd.to_datetime(df.index)
    
    # ... [Keep your existing plotting code for fig1, fig2, etc.] ...
    
    # Important: When calling plot(), ensure we use the index
    # Example for Fig 1:
    fig1, ax1 = plt.subplots(figsize=(12, 5))
    
    # Explicitly plot against index to enforce dates
    ax1.plot(df.index, df[price_col], label='Price', color='gray', linewidth=1.5)
    
    if not log.empty:
        # Ensure log dates are datetime objects
        log['entry_date'] = pd.to_datetime(log['entry_date'])
        log['exit_date'] = pd.to_datetime(log['exit_date'])
        
        ax1.scatter(log['entry_date'], log['entry_price'], marker='^', color='green', s=100, label='Buy', zorder=5)
        ax1.scatter(log['exit_date'], log['exit_price'], marker='v', color='red', s=100, label='Sell', zorder=5)
        
    # ... [Rest of your plotting code] ...
    """Generates and saves standard backtesting plots with improved aesthetics."""
    # Inside plot_backtest, ensuring date index
    if not isinstance(df.index, pd.DatetimeIndex):
        # Try to convert if 'Date' is a column
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date')
        
    if not os.path.exists(figures_dir):
        os.makedirs(figures_dir)
        
    log = result['trade_log']
    equity_mtm = result['equity_curve']
    equity_no_mtm = result['equity_curve_no_mtm']
    price_col = result['price_col_used']
    
    saved_plots = {}
    
    # 1. Price with Trades
    fig1, ax1 = plt.subplots(figsize=(12, 5))
    df[price_col].plot(ax=ax1, title='Price with Trades', legend=False, linewidth=1.5, color='gray')
    
    if not log.empty:
        buys = log.set_index('entry_date')
        sells = log.set_index('exit_date')
        
        # Increased markersize and added zorder=5 to ensure markers are on top of the line
        ax1.plot(buys.index, buys['entry_price'], '^', markersize=10, color='green', label='Buys (Entry)', zorder=5)
        ax1.plot(sells.index, sells['exit_price'], 'v', markersize=10, color='red', label='Sells (Exit)', zorder=5)

    # Improve date axis formatting and add grid
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax1.tick_params(axis='x', rotation=45)
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)
    fig1.tight_layout() # Ensures labels don't overlap
    fig1_path = os.path.join(figures_dir, 'price_with_trades.png')
    fig1.savefig(fig1_path)
    saved_plots['price_with_trades'] = fig1_path

    # 2. Equity Curve (MTM)
    fig2, ax2 = plt.subplots(figsize=(12, 5))
    equity_mtm.plot(ax=ax2, title='Equity Curve (Mark-to-Market)', legend=False, linewidth=2, color='blue')
    ax2.set_ylabel('Portfolio Value')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax2.tick_params(axis='x', rotation=45)
    fig2.tight_layout()
    fig2_path = os.path.join(figures_dir, 'equity_curve_mtm.png')
    fig2.savefig(fig2_path)
    saved_plots['equity_curve_mtm'] = fig2_path
    
    # 3. Equity Curve (No MTM)
    fig3, ax3 = plt.subplots(figsize=(12, 5))
    equity_no_mtm.plot(ax=ax3, title='Equity Curve (Closed Trades Only)', legend=False, color='orange', linewidth=2)
    ax3.set_ylabel('Portfolio Value')
    ax3.grid(True, linestyle='--', alpha=0.6)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax3.tick_params(axis='x', rotation=45)
    fig3.tight_layout()
    fig3_path = os.path.join(figures_dir, 'equity_curve_no_mtm.png')
    fig3.savefig(fig3_path)
    saved_plots['equity_curve_no_mtm'] = fig3_path

    # 4. Drawdown - Showing drawdown as percentage on Y-axis
    fig4, ax4 = plt.subplots(figsize=(12, 4))
    peak = equity_mtm.cummax()
    drawdown = (equity_mtm - peak) / peak
    
    max_dd_pct = result['metrics']['max_drawdown'] * 100
    ax4.set_title(f'Drawdown (Max Drawdown: {max_dd_pct:.2f}%)')
    
    drawdown.plot(ax=ax4, kind='area', alpha=0.7, color='crimson')
    ax4.axhline(0, color='black', linewidth=1)
    
    # Format Y-axis as percentage
    ax4.set_yticklabels(['{:.0f}%'.format(x * 100) for x in ax4.get_yticks()])
    ax4.grid(True, linestyle='--', alpha=0.6)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax4.tick_params(axis='x', rotation=45)
    fig4.tight_layout()
    fig4_path = os.path.join(figures_dir, 'drawdown.png')
    fig4.savefig(fig4_path)
    saved_plots['drawdown'] = fig4_path

    # 5. Histogram of Trade Returns
    fig5, ax5 = plt.subplots(figsize=(8, 4))
    if not log.empty:
        # Plot returns in percentage
        (log['return_pct'] * 100).hist(bins=30, ax=ax5, edgecolor='black', color='lightblue')
        mean_return = (log['return_pct'] * 100).mean()
        ax5.axvline(mean_return, color='orange', linestyle='dashed', linewidth=2, label=f'Mean Return: {mean_return:.2f}%')
        ax5.legend()
    ax5.set_title('Histogram of Trade Returns (%)')
    ax5.set_xlabel('Return (%)')
    ax5.set_ylabel('Frequency')
    ax5.grid(axis='y', linestyle='--', alpha=0.7)
    fig5.tight_layout()
    fig5_path = os.path.join(figures_dir, 'trade_returns_hist.png')
    fig5.savefig(fig5_path)
    saved_plots['trade_returns_hist'] = fig5_path

    # 6. Cumulative Returns
    fig6, ax6 = plt.subplots(figsize=(10, 4))
    if not log.empty:
        # Plot cumulative returns in percentage
        cum_returns = (log.set_index('exit_date')['return_pct'].sort_index().cumsum() * 100)
        cum_returns.plot(ax=ax6, title='Cumulative Trade Returns (%)', linewidth=2, color='purple')
    ax6.axhline(0, color='black', linestyle='--', linewidth=1)
    ax6.grid(True, linestyle='--', alpha=0.6)
    ax6.set_ylabel('Cumulative Return (%)')
    ax6.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax6.tick_params(axis='x', rotation=45)
    fig6.tight_layout()
    fig6_path = os.path.join(figures_dir, 'portfolio_cum_returns.png')
    fig6.savefig(fig6_path)
    saved_plots['portfolio_cum_returns'] = fig6_path


    # 7. Save trade log
    log_path = os.path.join(figures_dir, 'trade_log.csv')
    log.to_csv(log_path)
    saved_plots['trade_log_csv'] = log_path
    
    return {
        'plots': saved_plots,
        'fig_objs': {
            'price_with_trades': fig1,
            'equity_curve_mtm': fig2,
            'equity_curve_no_mtm': fig3,
            'drawdown': fig4,
            'trade_returns_hist': fig5,
            'cumulative_trade_returns': fig6
        }
    }

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates

def simulate_trading(df, predictions=None, initial_demat=20000.0, 
                     risk_per_trade=0.02,  # Risk 2% of EQUITY per trade
                     max_positions=5,      # Max concurrent trades
                     atr_multiplier=2.0):  # SL distance in ATRs
    """
    Advanced Backtester: Volatility-Adjusted Sizing + Trailing Stops + Equity Compounding.
    """
    
    # --- Setup ---
    if df.index.name != 'Date':
        if 'Date' in df.columns:
            df = df.set_index('Date').sort_index()
        else:
            raise ValueError("DataFrame must have a 'Date' column or index.")
            
    if predictions is None:
        if 'pred' not in df.columns:
            raise ValueError("df must contain 'pred' column.")
        signal_series = df['pred']
    else:
        signal_series = pd.Series(predictions, index=df.index)
        
    # Ensure columns exist
    price_col = 'raw_Close'
    high_col = 'raw_High'
    low_col = 'raw_Low'
    atr_col = 'raw_ATR_14' # Ensure this exists from feature engineering!
    
    # Check for ATR
    if atr_col not in df.columns:
        print("⚠️ ATR column missing. Calculating approx ATR...")
        df[atr_col] = (df[high_col] - df[low_col]).rolling(14).mean()

    demat_history = []
    cash = initial_demat
    open_positions = []
    trade_log = []
    trade_id_counter = 1
    
    df_reset = df.reset_index()

    # --- Main Loop ---
    for i, row in enumerate(df_reset.itertuples()):
        current_date = row.Date
        current_price = getattr(row, price_col)
        current_high = getattr(row, high_col)
        current_low = getattr(row, low_col)
        current_atr = getattr(row, atr_col)
        signal = signal_series.iloc[i]

        # 1. Mark-to-Market Update
        current_equity = cash
        positions_to_close = []
        
        for pos in open_positions:
            # Update trailing stop if price moves in our favor
            # Trail stop at: Highest High since entry minus (ATR * Multiplier)
            # Simple version: Trail at Entry Price + (Current Price - Entry Price) * 0.5 if profitable
            # Better version: Chandelier Exit logic
            
            # Dynamic Trailing Stop Logic
            new_stop = current_price - (current_atr * atr_multiplier)
            if new_stop > pos['stop_loss_price']:
                pos['stop_loss_price'] = new_stop
            
            exit_reason = None
            exit_price = current_price

            # Hit Stop Loss?
            if current_low <= pos['stop_loss_price']:
                exit_reason = 'SL'
                exit_price = pos['stop_loss_price'] 
            
            # Hit Time Exit? (Optional: remove time exit to ride trends longer)
            # elif i >= pos['exit_index']:
            #     exit_reason = 'TIME' 
            
            if exit_reason:
                pnl = (exit_price - pos['entry_price']) * pos['quantity']
                cash += (pos['capital_invested'] + pnl)
                
                trade_log.append({
                    'trade_id': pos['trade_id'],
                    'entry_date': pos['entry_date'],
                    'exit_date': current_date,
                    'entry_price': pos['entry_price'],
                    'exit_price': exit_price,
                    'pnl': pnl,
                    'return_pct': pnl / pos['capital_invested'],
                    'exit_reason': exit_reason,
                    'holding_days': i - pos['entry_index']
                })
                positions_to_close.append(pos)
            else:
                # Position still open
                pos_value = pos['quantity'] * current_price
                current_equity += pos_value

        # Clean up closed positions
        open_positions = [p for p in open_positions if p not in positions_to_close]
        
        demat_history.append({'Date': current_date, 'Demat': current_equity})

        # 2. Entry Logic (Aggressive Compounding)
        if signal == 1 and len(open_positions) < max_positions:
            # Position Sizing based on Risk
            # Risk Amount = Current Equity * Risk % (e.g., 2%)
            # Stop Loss Distance = ATR * Multiplier
            # Shares = Risk Amount / Stop Loss Distance
            
            risk_amount = current_equity * risk_per_trade
            sl_distance = current_atr * atr_multiplier
            
            # Safety check for zero ATR
            if sl_distance == 0: sl_distance = current_price * 0.02 
            
            qty = risk_amount / sl_distance
            capital_needed = qty * current_price
            
            # Allow trade if we have enough cash
            if cash >= capital_needed:
                cash -= capital_needed
                stop_loss = current_price - sl_distance
                
                open_positions.append({
                    'trade_id': trade_id_counter,
                    'entry_index': i,
                    'entry_date': current_date,
                    'entry_price': current_price,
                    'quantity': qty,
                    'stop_loss_price': stop_loss,
                    'capital_invested': capital_needed,
                    'pred_at_entry': signal,
                    'available_cash_before_entry': cash + capital_needed,
                    'raw_20d_fwd_return_used': 0 
                })
                trade_id_counter += 1

    # Close remaining at end
    if open_positions:
        last_row = df_reset.iloc[-1]
        for pos in open_positions:
            pnl = (last_row.raw_Close - pos['entry_price']) * pos['quantity']
            cash += (pos['capital_invested'] + pnl)
            trade_log.append({
                'trade_id': pos['trade_id'],
                'entry_date': pos['entry_date'],
                'exit_date': last_row.Date,
                'entry_price': pos['entry_price'],
                'exit_price': last_row.raw_Close,
                'pnl': pnl,
                'return_pct': pnl / pos['capital_invested'],
                'exit_reason': 'EOD',
                'holding_days': len(df_reset) - pos['entry_index']
            })

    # Metrics
    final_demat = cash
    log_df = pd.DataFrame(trade_log)
    equity_df = pd.DataFrame(demat_history).set_index('Date')['Demat']
    
    # Simple No-MTM Curve
    equity_no_mtm = pd.Series(initial_demat, index=df.index, name='Demat_No_MTM')
    if not log_df.empty:
        daily_pnl = log_df.groupby('exit_date')['pnl'].sum()
        pnl_cumsum = daily_pnl.sort_index().cumsum()
        equity_no_mtm = (equity_no_mtm + pnl_cumsum.reindex(equity_no_mtm.index, method='ffill')).fillna(method='ffill')

    metrics = calculate_metrics(log_df, initial_demat, final_demat, equity_df)
    
    return {
        'final_demat': final_demat,
        'trade_log': log_df,
        'equity_curve': equity_df,
        'equity_curve_no_mtm': equity_no_mtm,
        'metrics': metrics,
        'price_col_used': price_col
    }

In [ ]:
def simulate_trading_1(df, predictions=None, initial_demat=20000.0, 
                     base_risk_per_trade=0.02,  # Increased base risk slightly
                     max_concurrent_trades=5,   # Increased from 3 to 5
                     atr_multiplier=2.0, 
                     cooldown_days=1):          # Reduced from 3 to 1 for more activity
    """
    Regime-Adaptive Backtester: Fixed Visualization Key + Tuned Aggression.
    """
    
    # --- Setup ---
    if df.index.name != 'Date':
        if 'Date' in df.columns:
            df = df.set_index('Date').sort_index()
        else:
            raise ValueError("DataFrame must have a 'Date' column or index.")
            
    if predictions is None:
        signal_series = df['pred']
    else:
        signal_series = pd.Series(predictions, index=df.index)
        
    price_col = 'raw_Close'
    high_col = 'raw_High'
    low_col = 'raw_Low'
    atr_col = 'raw_ATR_14'
    
    if atr_col not in df.columns:
        df[atr_col] = (df[high_col] - df[low_col]).rolling(14).mean()

    # --- State ---
    cash = initial_demat
    open_positions = []
    trade_log = []
    demat_history = []
    
    trade_id_counter = 1
    last_entry_index = -999
    consecutive_losses = 0
    
    df_reset = df.reset_index()

    # --- Main Loop ---
    for i, row in enumerate(df_reset.itertuples()):
        current_date = row.Date
        current_price = getattr(row, price_col)
        current_low = getattr(row, low_col)
        current_high = getattr(row, high_col)
        current_atr = getattr(row, atr_col)
        signal = signal_series.iloc[i]
        
        current_equity = cash
        positions_to_close = []
        
        # 1. Manage Positions
        for pos in open_positions:
            # Mark to Market
            pos_value = pos['qty_remaining'] * current_price
            current_equity += pos_value
            
            # A. Profit Taking (+3 ATR) - Banks profit, keeps runner
            if not pos['profit_taken']:
                target_price = pos['entry_price'] + (3 * current_atr)
                if current_high >= target_price:
                    qty_to_sell = int(pos['qty_remaining'] / 2) # Sell half
                    if qty_to_sell > 0:
                        cash_in = qty_to_sell * target_price
                        cash += cash_in
                        pos['qty_remaining'] -= qty_to_sell
                        pos['profit_taken'] = True
                        # Move SL to Breakeven to protect the runner
                        pos['stop_loss_price'] = max(pos['stop_loss_price'], pos['entry_price'])
            
            # B. Trailing Stop (Aggressive trail on runners)
            if pos['profit_taken']:
                # Tight trail (1.5 ATR) to lock in runner gains
                new_stop = current_price - (current_atr * 1.5)
            else:
                # Loose trail (2.5 ATR) to give room to breathe initially
                new_stop = current_price - (current_atr * 2.5)
                
            if new_stop > pos['stop_loss_price']:
                pos['stop_loss_price'] = new_stop
            
            # C. Check Exit
            exit_reason = None
            exit_price = current_price
            
            if current_low <= pos['stop_loss_price']:
                exit_reason = 'SL' if not pos['profit_taken'] else 'Trailing_Stop'
                exit_price = pos['stop_loss_price']
            
            if exit_reason:
                cash_in = pos['qty_remaining'] * exit_price
                cash += cash_in
                
                # Approx PnL for log (Exit Value - Cost Basis of remaining)
                # Note: This is a simplified view for the log; Cash is the source of truth
                trade_pnl = (exit_price - pos['entry_price']) * pos['qty_remaining']
                
                if trade_pnl > 0:
                    consecutive_losses = 0
                else:
                    consecutive_losses += 1
                
                trade_log.append({
                    'trade_id': pos['trade_id'],
                    'entry_date': pos['entry_date'],
                    'exit_date': current_date,
                    'pnl': trade_pnl, 
                    'return_pct': trade_pnl / (pos['entry_price'] * pos['initial_qty']),
                    'exit_reason': exit_reason,
                    'holding_days': i - pos['entry_index']
                })
                positions_to_close.append(pos)

        open_positions = [p for p in open_positions if p not in positions_to_close]
        demat_history.append({'Date': current_date, 'Demat': current_equity})
        
        # 2. Entry Logic
        # Adjust risk based on streak
        current_risk_pct = base_risk_per_trade
        if consecutive_losses >= 2:
            current_risk_pct = base_risk_per_trade / 2
        
        if signal == 1 and len(open_positions) < max_concurrent_trades:
            if (i - last_entry_index) > cooldown_days:
                
                risk_amt = current_equity * current_risk_pct
                sl_dist = current_atr * atr_multiplier
                if sl_dist == 0: sl_dist = current_price * 0.02
                
                # Calculate quantity
                qty = int(risk_amt / sl_dist)
                if qty < 1: qty = 1 # Minimum 1 share
                
                capital_needed = qty * current_price
                
                if cash >= capital_needed:
                    cash -= capital_needed
                    stop_loss = current_price - sl_dist
                    
                    open_positions.append({
                        'trade_id': trade_id_counter,
                        'entry_index': i,
                        'entry_date': current_date,
                        'entry_price': current_price,
                        'qty_remaining': qty,
                        'initial_qty': qty,
                        'initial_capital': capital_needed,
                        'stop_loss_price': stop_loss,
                        'profit_taken': False,
                        'cash_snapshot_at_entry': cash
                    })
                    
                    trade_id_counter += 1
                    last_entry_index = i

    # Close remaining
    if open_positions:
        last_row = df_reset.iloc[-1]
        for pos in open_positions:
            cash_in = pos['qty_remaining'] * last_row.raw_Close
            cash += cash_in
            # Log trade...

    metrics = calculate_metrics(pd.DataFrame(trade_log), initial_demat, cash, pd.DataFrame(demat_history).set_index('Date')['Demat'])
    
    return {
        'final_demat': cash,
        'trade_log': pd.DataFrame(trade_log),
        'equity_curve': pd.DataFrame(demat_history).set_index('Date')['Demat'],
        'equity_curve_no_mtm': pd.DataFrame(demat_history).set_index('Date')['Demat'], 
        'metrics': metrics,
        'price_col_used': price_col  # <--- FIX: Added this key back!
    }